# Deep Agents Webinar: Journal Agent

This notebook builds one agent, then extends it: a real sentiment analysis tool, a working human-in-the-loop approval flow, and subagents that delegate work to specialists.

For topics not covered today, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langgraph langchain-anthropic vaderSentiment dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()

# Add your ANTHROPIC_API_KEY to a .env file in this directory

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("anthropic:claude-sonnet-4-5")

# To use a different provider instead, replace the line above, for example:
# model = init_chat_model("openai:gpt-4.1")

## 1: The harness, what you get before writing any tool code

Every deep agent starts the same way: a model wrapped in a harness that already knows how to read and write files, plan, and call tools. No tools are added yet. The next cell shows what it can already do.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model)

result = agent.invoke({"messages": [{"role": "user", "content": (
    "Start a journal.md file. Log a new dated entry from these notes:\n"
    "- What I learned today: how to build a deep agent\n"
    "- How I felt: excited but a little overwhelmed\n"
    "- What's next: add a custom tool\n"
    "Then read the file back to me."
)}]})

for m in result['messages']:
    m.pretty_print()


## 2: Set its role with a system prompt

One `system_prompt` string controls the voice the agent writes in, on top of whatever facts you give it. The agent still writes the entry itself: it takes your notes as raw facts and composes them into sentences, based on the persona you give it.

In [ ]:
system_prompt = ""  # baseline: no persona

# Other personas to demo:
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a toddler. Explain everything like you're five."
# system_prompt = "You are a melodramatic Victorian child. Narrate everything with excessive despair and flowery, dramatic language."

agent = create_deep_agent(model=model, system_prompt=system_prompt)

result = agent.invoke({"messages": [{"role": "user", "content":
    "Log a three-sentence journal entry from these notes: what I learned today "
    "(how to build a deep agent), how I felt (excited but a little overwhelmed), "
    "what's next (add a custom tool)."
}]})

for m in result['messages']:
    m.pretty_print()


In [ ]:
# Reset the persona before moving on, so a persona you tried above
# doesn't leak into the sections below.
system_prompt = ""

## 3: Give it a custom tool

A plain Python function becomes a tool the agent can call. Tools are how an agent's abilities grow past reading and writing files, this is the piece you'll customize most often.

In [ ]:
from langchain_core.tools import tool
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

@tool
def word_count(text: str) -> str:
    """Count the words in a piece of text."""
    return f"{len(text.split())} words"

# vaderSentiment ships its lexicon inside the package, so this needs no
# runtime download, unlike nltk's VADER (nltk.download("vader_lexicon")) or
# textblob (python -m textblob.download_corpora).

_sentiment_analyzer = SentimentIntensityAnalyzer()

@tool
def mood_tag(text: str) -> str:
    """Tag a piece of text with a mood, using real sentiment analysis."""
    compound = _sentiment_analyzer.polarity_scores(text)["compound"]
    if compound >= 0.5:
        mood = "positive"
    elif compound <= -0.5:
        mood = "negative"
    else:
        mood = "neutral"
    return f"{mood} (compound score: {compound:.2f})"

TOOL_MENU = {
    "word_count": word_count,
    "mood_tag": mood_tag,
}

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back


**Try it:** the cell below picks `mood_tag` from the menu, real sentiment analysis instead of a canned string, and runs it on a journal entry.


In [ ]:
chosen_tool = TOOL_MENU["mood_tag"]

agent = create_deep_agent(model=model, system_prompt=system_prompt, tools=[chosen_tool])

result = agent.invoke({"messages": [{"role": "user", "content":
    "Here is a journal entry: 'Today I finally shipped the feature I've been stuck on for a "
    "week. The bug turned out to be a caching issue that took forever to track down, and I "
    "ended up rewriting most of the retry logic to fix it. It feels good to have it done, "
    "though I'm a little worried about whether the fix will hold up under real traffic. "
    "Tomorrow I want to write better tests before touching anything else.' "
    "Use your tool on it, then tell me what you found."
}]})

for m in result['messages']:
    m.pretty_print()


## OPTIONAL: Give it memory across turns

Short-term memory for any agent built with LangChain, including Deep Agents, is managed using a **checkpointer**. It's what lets an agent remember earlier turns in the same conversation, tracked by a `thread_id`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "journal-memory-demo"}}

agent = create_deep_agent(model=model, system_prompt=system_prompt, checkpointer=checkpointer)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Jess."}]},
    config=config,
)

for m in result['messages']:
    m.pretty_print()

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    config=config,
)

for m in result['messages']:
    m.pretty_print()

## 4: Human-in-the-loop, approve a risky action before it happens

`interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes. Any agent with access to money, message sends, or irreversible actions needs this kind of control before it is used in production.

This only works because a checkpointer is attached to the agent: it saves the agent's state at the interrupt point so the run can be resumed later, potentially after the human has stepped away and come back. The cells below build a real example: a `share_journal_entry` tool that requires approval before it runs.

In [ ]:
from langgraph.types import Command

@tool
def share_journal_entry(entry: str, platform: str) -> str:
    """Share a journal entry to an external platform. This simulates a send, no network call is made."""
    return f"Shared to {platform}: {entry[:60]}..."

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[chosen_tool, share_journal_entry],
    interrupt_on={"share_journal_entry": True},
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-hitl-demo"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Start a journal.md file. Log a new dated entry: 'Set up human-in-the-loop "
        "approval today, it feels reassuring to have a real gate before anything gets "
        "shared externally.' Then read the file back, and share the most recent entry "
        "to the 'team-standup' platform."
    )}]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    result["messages"][-1].pretty_print()

The cell above paused instead of finishing, because `share_journal_entry` matched `interrupt_on`. The cell below resumes it with an approval decision.


In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)

for m in result['messages']:
    m.pretty_print()

## 5: Subagents, delegate to a specialist

`subagents` is a list of specialized sub-agents the main agent can hand work off to through a built-in `task` tool. Each one gets its own system prompt and its own isolated context window, so a specialized job runs without cluttering the main agent's context. With more than one subagent defined, the main agent picks between them based on nothing but each subagent's `description`.

Each entry in `subagents` is a plain dict: a `name`, a `description` (what the main agent reads to decide when to delegate), and a `system_prompt`. If `tools` is left out, as it is for both subagents below, the subagent inherits the main agent's tools; every subagent also gets the same filesystem tools (`read_file`, `grep`, `write_file`, and so on) regardless of what `tools` says.

The cells below define two subagents, then seed a week of journal entries with the same `checkpointer` pattern from the human-in-the-loop section, so both subagents have real, repeated data to work from instead of a single one-off entry:

- `life-planner` greps the whole journal for a complaint that keeps showing up across entries and turns the backlog into a weekly plan.
- `devils-advocate` surfaces the practical downsides of a decision mentioned in the journal.

In [ ]:
life_planner = {
    "name": "life-planner",
    "description": (
        "Use this specific subagent, not a general-purpose one, whenever the user says "
        "they feel overwhelmed/burnt out or explicitly asks for help getting organized "
        "(for example: 'help me get organized', 'I'm overwhelmed', 'make me a plan'). "
        "It reads the entire journal.md, finds any complaint or chore that repeats "
        "across multiple entries, and turns that backlog into a structured weekly plan."
    ),
    "system_prompt": (
        "You are a life planner. Read journal.md in full. Use grep to check whether "
        "any complaint or chore (being tired, a specific errand) shows up in more "
        "than one entry. Write a short weekly plan to planner.md: name any pattern "
        "you noticed directly (for example, 'you've mentioned being tired 3 days "
        "running'), suggest one concrete change for the most repeated chore (for "
        "example, sending laundry out instead of doing it yourself), then lay out "
        "the rest of the obligations mentioned across the entries as a simple "
        "day-by-day list. Return the plan as your answer, not just the file."
    ),
}

devils_advocate = {
    "name": "devils-advocate",
    "description": (
        "Use this specific subagent, not a general-purpose one, whenever the user is "
        "weighing or asks about a decision (for example: 'should I...', 'is it worth "
        "it to...'), not just venting. It reads journal.md, finds the decision under "
        "consideration, and lists the concrete practical downsides (cost, time, "
        "logistics, anything that could go wrong) as a short list."
    ),
    "system_prompt": (
        "You are a practical, slightly skeptical friend. Read journal.md, find the "
        "decision the user is weighing, then list the concrete practical "
        "considerations they'd need to deal with (cost, time, logistics, anything "
        "that could go wrong) as a short list. Don't tell them what to decide, just "
        "make sure they've seen the unglamorous side before they commit."
    ),
}

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    subagents=[life_planner, devils_advocate],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-subagent-demo"}}

seed_entries = (
    "## Day 1\nI'm exhausted today. So much to do at work and I haven't done "
    "laundry in two weeks.\n\n"
    "## Day 2\nAnother tiring day. Skipped the gym again. Still need to renew my "
    "license.\n\n"
    "## Day 3\nFeeling overwhelmed. Work is piling up, forgot to take my vitamins "
    "again, and the laundry pile keeps growing.\n\n"
    "## Day 4\nBig news: my job is going fully remote starting next month. I've "
    "been thinking about getting a cat since I'll be home so much more. I think it "
    "would make me happy!\n\n"
    "## Day 5\nStill tired. Still haven't touched the laundry. Keep thinking about "
    "that cat, might visit a shelter this weekend."
)

agent.invoke(
    {"messages": [{"role": "user", "content":
        f"Log these journal entries to journal.md, each under its own heading:\n\n{seed_entries}"
    }]},
    config=config,
)

print("Journal seeded.")

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "I'm so overwhelmed with everything I need to do, can you help me get organized?"
    }]},
    config=config,
)

for m in result['messages']:
    m.pretty_print()


The plan above should call out the tiredness and laundry mentions by name, since both show up across more than one entry, something only visible by reading the whole journal at once, which is exactly the kind of job worth delegating instead of doing inline.

The next message hands the same agent, same thread, a decision instead of a backlog, which should route to `devils-advocate` instead.

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Should I get a cat?"}]},
    config=config,
)

for m in result['messages']:
    m.pretty_print()


## Wrap-up

We built: a filesystem-backed agent, a way to swap personas, a custom tool with real sentiment analysis, short-term memory across turns, a human-in-the-loop approval flow, and two subagents the main agent picks between on its own (one of which writes an explicit, visible plan instead of leaving it implicit in a chat reply).

In the full LangChain Academy Deep Agents course: we cover planning middleware, backends (filesystem/store/composite), skills, memory, sandboxes, deployment, and many more things!